# L11a: Foundations of Markov Models and Markov Chains
In this lecture, we explore Markov models, one of the most useful tools for modeling systems with randomness. These models appear everywhere, from operations and finance to biology and machine learning.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the Markov property:__ A random process has the Markov property when the probability of its next state depends only on its present state. We'll see why this memoryless property makes random systems much easier to model and analyze.
> * __Build a transition matrix:__ The transition matrix holds the probability of moving from each state to every state in one time step. We'll describe the properties that every row of a valid transition matrix must satisfy.
> * __Compute the stationary distribution:__ The stationary distribution remains unchanged under the transition matrix. For an ergodic chain, it also describes the long-run fraction of time spent in each state. We'll compute it with the power iteration method, which applies the transition matrix over and over until the distribution stops changing.

Markov models can capture real-world (stochastic) dynamics well when the present state contains all relevant information. Let's get started!
___

## Discrete Markov Model States
A discrete Markov model is a stochastic (random) process that describes the transitions between states based on defined probabilities. Formally, a Markov model is determined by the tuple $\mathcal{M} = (\mathcal{S},\mathbf{P})$, where $\mathcal{S}$ represents the set of states in the model and $\mathbf{P}$ is the transition matrix.

### The state space $\mathcal{S}$
The state space $\mathcal{S}$ is the set of all possible values a system can assume. For example, if a Markov chain can be in the states $\left\{1,2,3\right\}$, then $\mathcal{S} \equiv \left\{1,2,3\right\}$. What do these states represent? The answer depends on the problem. Here are two concrete examples:
* __Letters in words__ $\mathcal{S} \equiv \left\{a,b,c,\dotsc,z\right\}$: If the state space $\mathcal{S}$ is defined as the alphabet, we could develop a Markov model to generate $n$ words, each of length $l$ characters, that start with a specific letter such as $t$.
* __Investor moods__ $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$: In this case, the state space is defined as three possible moods an investor can experience. Using this type of $\mathcal{S}$, we could build a Markov model to simulate how an investor's mood changes as they watch the market, the news, and other factors.

Next, let's look at the Markov property and transition matrix $\mathbf{P}$.
___

## The Markov Property
The Markov property describes a random process whose next-state probabilities depend only on its present state. Choosing a state that contains the information needed to predict the next step lets us describe the process using one-step transition probabilities.

> __Definition:__ Let $X_n$ denote the state of a stochastic process at time step $n$. The process satisfies the **Markov property** if the following equality holds for every $n\geq 0$ and every state history with positive probability:
>
> $$
> P(X_{n+1}=s_{n+1}\mid X_0=s_0,\ldots,X_n=s_n)
> =P(X_{n+1}=s_{n+1}\mid X_n=s_n).
> $$
>
> Given the present state, the next state is conditionally independent of the past states. This lets us update the state distribution without storing the full history of the process.

### One-Step Transition Probabilities
For a finite state space $\mathcal{S}=\{s_1,\ldots,s_N\}$, the probability of moving from state $s_i$ to state $s_j$ in one step is given by:

$$
p_{ij}=P(X_{n+1}=s_j\mid X_n=s_i).
$$

We assume the process is __time-homogeneous__, meaning these probabilities stay constant over time. Thus, $p_{ij}$ does not depend on $n$, and the same transition matrix applies at every step.

### Why the Markov Property Matters
The Markov property simplifies our modeling work: instead of tracking the entire history of a process, we only need to know the present state. Without this property, the number of possible historical sequences would grow exponentially. With it, the problem becomes tractable and easier to work with.

___

## The Transition Matrix
We collect the one-step probabilities in the transition matrix $\mathbf{P}\in\mathbb{R}^{N\times N}$. Row $i$ describes the next-state distribution when the current state is $s_i$:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-TransitionMatrix-Schematic.svg" width="680" style="max-width:100%; height:auto;" alt="Schematic of a transition matrix drawn as a grid of dots inside square brackets. A shaded row is labeled rows equal current states, with the note sum equals 1. A shaded column is labeled columns equal future states. A red dot in row one, column two is labeled p12, and a yellow dot on the diagonal is labeled pii, a self-loop."/>
</p>

The transition matrix has the following properties:
* Every entry is non-negative: $p_{ij}\geq 0$.
* Rows represent the current states, and columns represent the next states (our convention). Entry $p_{ij}$ is the probability of moving from $s_i$ to $s_j$.
* Every row sums to one: $\sum_{j=1}^{N}p_{ij}=1$. Starting from any state, the chain must move to one of the states in $\mathcal{S}$, including the possibility of staying where it is.

___

## Algorithm: Power Iteration Method for the Stationary Distribution
The stationary distribution $\bar{\pi}$ of a Markov chain with transition matrix $\mathbf{P}$ is a probability distribution that remains unchanged as the system evolves, $\bar{\pi} = \bar{\pi}\mathbf{P}$. Here, each distribution $\pi$ is a row vector whose $j$-th element is the probability that the chain is in state $s_{j}$. 

> __Interesting analogy__ 
>
> The stationary distribution is analogous to the steady-state distribution of a physical system, where the system's properties do not change over time. In an ergodic Markov chain (every state can be reached from every other state, and the chain is not locked into a cycle), the stationary distribution describes the long-run fraction of time spent in each state.

We estimate $\bar{\pi}$ by repeatedly applying the transition matrix to an initial distribution until convergence:

$$
\begin{align*}
\pi_1 &= \pi_0 \mathbf{P} \\
\pi_2 &= \pi_1 \mathbf{P} = \pi_0 \mathbf{P}^2 \\
\pi_3 &= \pi_2 \mathbf{P} = \pi_0 \mathbf{P}^3 \\
&\vdots \\
\pi_k &= \pi_{k-1} \mathbf{P} = \pi_0 \mathbf{P}^k
\end{align*}
$$

__Initialize__: Given an ergodic transition matrix $\mathbf{P}$ with $N$ states, a tolerance $\epsilon>0$, and a maximum number of iterations $T$, initialize the iteration counter $t\gets 1$, the uniform initial distribution $\pi \gets \left[1/N,\ldots,1/N\right]$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. Compute: $\pi^{\prime} \gets \pi\mathbf{P}$.
2. Check whether to stop:
    - If $\lVert \pi^{\prime} - \pi \rVert_{1} \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and return $\bar{\pi}\gets\pi^{\prime}$.
    - Otherwise, if $t\geq{T}$, then return $\pi^{\prime}$ with $\texttt{converged}=\texttt{false}$ and a warning that the maximum number of iterations was reached without convergence.
3. Update the distribution: $\pi\gets\pi^{\prime}$.
4. Increment the iteration counter: $t\gets{t+1}$.

__Output__: The stationary distribution $\bar{\pi}$.

The convergence rate depends on the spectral gap (difference between the largest and second-largest eigenvalue magnitudes of $\mathbf{P}$), with larger gaps enabling faster convergence.

__Notes:__
1. This algorithm assumes $\mathbf{P}$ is ergodic, that is, every state can be reached from every other state and aperiodic (the chain is not locked into a cycle, such as returning to a state only on every second step). An ergodic chain has a unique stationary distribution, and power iteration converges to it from any initial distribution.
2. The $\pi^{\prime} \gets \pi\mathbf{P}$ step is a left vector-matrix multiplication, updating the distribution based on transition probabilities. See below for a schematic.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Matrix-Vector-Left-bA-product.svg" width="580" style="max-width:100%; height:auto;" alt="A row vector with elements a, b, c multiplies a three-by-three matrix whose rows are red, green, and blue. The product equals a times the red row, plus b times the green row, plus c times the blue row, which gives a single row vector."/>
</p>

___

## Example
Let's explore a simple Markov chain, compute its stationary distribution, and experiment with the model parameters to see how they affect the long-term behavior.

> [▶ Let's explore properties of Markov models and stationary distributions](CHEME-5800-L11a-Example-MarkovModels-Fall-2026.ipynb). In this example, you will explore the fundamental properties of Markov models by constructing a discrete-time Markov chain, computing its stationary distribution, and validating the theoretical predictions through simulation.

___

## Lab
In the [lab](../L11b/CHEME-5800-L11b-Lab-MarkovWordGenerator-Fall-2026.ipynb), we build a Markov model that generates words one character at a time, using letter transition probabilities estimated from a list of about 370,000 English words.

___

## Summary
In this lecture, we explored how Markov models capture the dynamics of systems where the future depends only on the present state:

> __Key Takeaways:__
>
> * __The Markov property:__ We assumed that the next state of a random process depends only on its present state, not on the path that led there. This let us describe the process with one-step transition probabilities instead of its entire history.
>
> * __The transition matrix:__ We packed the one-step transition probabilities into a matrix whose rows are the current states and whose columns are the next states. Every element is non-negative and every row sums to one, and for a time-homogeneous chain the same matrix applies at every step.
>
> * __The stationary distribution:__ We computed the long-run distribution of an ergodic chain with power iteration, applying the transition matrix to a distribution until it stopped changing. The result tells us the fraction of time the chain spends in each state after many steps.

These foundational concepts enable practical applications across operations, finance, and machine learning where systems evolve based on discrete states and probabilistic transitions.

___